In [27]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
import json
import os
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cpu')

# Load the feature data
df = pd.read_csv("../data/processed/features.csv")
df = df.sort_values(['Email', 'assignment_order']).reset_index(drop=True)

print(f"Loaded features: {df.shape}")
print(f"Students: {df['Email'].nunique()}")
print(f"Assignments: {df['Assignment_Name'].nunique()}")

Loaded features: (588, 18)
Students: 42
Assignments: 14


In [28]:
class AcademicGRU(nn.Module):
    def __init__(self, input_size=13, hidden_size=64,
                 num_layers=2, dropout=0.3):
        super(AcademicGRU, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers  = num_layers
        self.gru = nn.GRU(input_size=input_size, hidden_size=hidden_size,
                          num_layers=num_layers, batch_first=True,
                          dropout=dropout if num_layers > 1 else 0.0)
        self.dropout   = nn.Dropout(dropout)
        self.attention = nn.Linear(hidden_size, 1)
        self.grade_head = nn.Sequential(
            nn.Linear(hidden_size, 32), nn.ReLU(),
            nn.Dropout(0.2), nn.Linear(32, 1))
        self.risk_head = nn.Sequential(
            nn.Linear(hidden_size, 32), nn.ReLU(),
            nn.Dropout(0.2), nn.Linear(32, 1), nn.Sigmoid())

    def forward(self, x):
        batch_size = x.size(0)
        h0 = torch.zeros(self.num_layers, batch_size, self.hidden_size)
        gru_out, _ = self.gru(x, h0)
        gru_out = self.dropout(gru_out)
        attn_scores  = self.attention(gru_out)
        attn_weights = torch.softmax(attn_scores, dim=1)
        context      = (attn_weights * gru_out).sum(dim=1)
        grade_pred   = self.grade_head(gru_out).squeeze(-1)
        risk_pred    = self.risk_head(context).squeeze(-1)
        return grade_pred, risk_pred, attn_weights.squeeze(-1)


class AcademicCNNGRU(nn.Module):
    def __init__(self, input_size=13, cnn_channels=64,
                 hidden_size=64, num_layers=2, dropout=0.3):
        super(AcademicCNNGRU, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers  = num_layers
        self.conv1d   = nn.Conv1d(input_size, cnn_channels, kernel_size=3, padding=1)
        self.conv_bn  = nn.BatchNorm1d(cnn_channels)
        self.conv_act = nn.ReLU()
        self.conv_drop = nn.Dropout(dropout)
        self.gru = nn.GRU(input_size=cnn_channels, hidden_size=hidden_size,
                          num_layers=num_layers, batch_first=True,
                          dropout=dropout if num_layers > 1 else 0.0)
        self.dropout   = nn.Dropout(dropout)
        self.attention = nn.Linear(hidden_size, 1)
        self.grade_head = nn.Sequential(
            nn.Linear(hidden_size, 32), nn.ReLU(),
            nn.Dropout(0.2), nn.Linear(32, 1))
        self.risk_head = nn.Sequential(
            nn.Linear(hidden_size, 32), nn.ReLU(),
            nn.Dropout(0.2), nn.Linear(32, 1), nn.Sigmoid())

    def forward(self, x):
        x_conv = x.transpose(1, 2)
        x_conv = self.conv_drop(self.conv_act(self.conv_bn(self.conv1d(x_conv))))
        x_conv = x_conv.transpose(1, 2)
        batch_size = x_conv.size(0)
        h0 = torch.zeros(self.num_layers, batch_size, self.hidden_size)
        gru_out, _ = self.gru(x_conv, h0)
        gru_out = self.dropout(gru_out)
        attn_scores  = self.attention(gru_out)
        attn_weights = torch.softmax(attn_scores, dim=1)
        context      = (attn_weights * gru_out).sum(dim=1)
        grade_pred   = self.grade_head(gru_out).squeeze(-1)
        risk_pred    = self.risk_head(context).squeeze(-1)
        return grade_pred, risk_pred, attn_weights.squeeze(-1)


print("Model classes defined.")

Model classes defined.


In [29]:
# Load GRU v1 (best classifier)
checkpoint_v1 = torch.load("../outputs/models/gru_v1.pth",
                            map_location='cpu', weights_only=False)
gru_v1 = AcademicGRU(**checkpoint_v1['model_config'])
gru_v1.load_state_dict(checkpoint_v1['model_state_dict'])
gru_v1.eval()

# Load CNN-GRU v2 (best regressor)
checkpoint_v2 = torch.load("../outputs/models/cnn_gru_v2.pth",
                            map_location='cpu', weights_only=False)
cnn_gru_v2 = AcademicCNNGRU(**checkpoint_v2['model_config'])
cnn_gru_v2.load_state_dict(checkpoint_v2['model_state_dict'])
cnn_gru_v2.eval()

# Use the scaler from GRU v1 checkpoint
scaler = checkpoint_v1['scaler']

print("GRU v1 loaded  (best classifier: 100% accuracy)")
print("CNN-GRU v2 loaded  (best regressor: MAE 7.05)")

GRU v1 loaded  (best classifier: 100% accuracy)
CNN-GRU v2 loaded  (best regressor: MAE 7.05)


In [30]:
FEATURE_COLS = checkpoint_v1['feature_cols']

# Scale features using saved scaler
df_scaled = df.copy()
df_scaled[FEATURE_COLS] = scaler.transform(df[FEATURE_COLS])

# Assignment order mapping for labels
assignment_names = (df.sort_values('assignment_order')
                      ['Assignment_Name'].unique().tolist())

all_student_results = []

for student_email, group in df_scaled.groupby('Email'):
    group = group.sort_values('assignment_order')

    # Build input tensor [1, 19, 13]
    features = torch.tensor(
        group[FEATURE_COLS].values.astype(np.float32)
    ).unsqueeze(0)

    with torch.no_grad():
        # GRU v1 — use for risk classification + attention
        _, risk_prob_v1, attn_v1 = gru_v1(features)
        risk_probability = float(risk_prob_v1[0].item())
        at_risk_pred = int(risk_probability > 0.5)
        attention_weights = attn_v1[0].numpy().tolist()

        # CNN-GRU v2 — use for grade prediction
        grade_preds_v2, _, _ = cnn_gru_v2(features)
        predicted_next_grades = (grade_preds_v2[0].numpy() * 100).tolist()

    # Actual grades from raw data
    actual_grades = group['grade_pct'].tolist()
    assignments   = group['Assignment_Name'].tolist()
    orders        = group['assignment_order'].tolist()

    # True label
    true_at_risk  = int(group['at_risk'].iloc[0])
    final_avg     = float(group['grade_pct'].mean())
    
    # Cumulative stats
    cumulative_skips = int(group['never_started'].sum())

    # Most attended assignment (highest attention weight)
    top_attn_idx  = int(np.argmax(attention_weights))
    top_attn_name = assignments[top_attn_idx] if top_attn_idx < len(assignments) else "Unknown"

    all_student_results.append({
        "email": student_email,
        "final_avg_grade": round(final_avg, 2),
        "at_risk_predicted": at_risk_pred,
        "at_risk_actual": true_at_risk,
        "risk_probability": round(risk_probability * 100, 1),
        "total_skips": cumulative_skips,
        "top_attended_assignment": top_attn_name,
        "grade_trajectory": [
            {
                "assignment_order": int(o),
                "assignment_name": a,
                "actual_grade": round(g, 1),
                "predicted_next_grade": round(p, 1)
            }
            for o, a, g, p in zip(
                orders, assignments,
                actual_grades, predicted_next_grades
            )
        ],
        "attention_weights": [
            round(float(w), 4) for w in attention_weights
        ]
    })

print(f"Generated predictions for {len(all_student_results)} students")
print(f"\nSample output for first student:")
s = all_student_results[0]
print(f"  Email:           {s['email']}")
print(f"  Final average:   {s['final_avg_grade']}%")
print(f"  At risk (pred):  {s['at_risk_predicted']}")
print(f"  Risk probability: {s['risk_probability']}%")
print(f"  Total skips:     {s['total_skips']}")
print(f"  Top assignment:  {s['top_attended_assignment']}")

Generated predictions for 42 students

Sample output for first student:
  Email:           afarooq8@gmu.edu
  Final average:   0.12%
  At risk (pred):  1
  Risk probability: 99.9%
  Total skips:     -2
  Top assignment:  IT207-Lab9-SQL_Review


In [31]:
# Assignment difficulty scores already computed in features.csv
difficulty_data = (
    df[['Assignment_Name', 'assignment_order', 'difficulty_score',
        'class_skip_rate']]
    .drop_duplicates('Assignment_Name')
    .sort_values('assignment_order')
)

# Class mean grade per assignment
class_means = (df.groupby('Assignment_Name')['grade_pct']
                 .mean()
                 .rename('class_mean_grade')
                 .reset_index())
difficulty_data = difficulty_data.merge(class_means, on='Assignment_Name')

assignment_difficulty = []
for _, row in difficulty_data.iterrows():
    assignment_difficulty.append({
        "assignment_order": int(row['assignment_order']),
        "assignment_name": row['Assignment_Name'],
        "difficulty_score": round(float(row['difficulty_score']), 2),
        "skip_rate_pct": round(float(row['class_skip_rate']) * 100, 1),
        "class_mean_grade": round(float(row['class_mean_grade']), 1)
    })

print(f"Assignment difficulty data: {len(assignment_difficulty)} assignments")
print("\nTop 5 hardest assignments:")
hardest = sorted(assignment_difficulty,
                 key=lambda x: x['difficulty_score'], reverse=True)[:5]
for a in hardest:
    print(f"  {a['difficulty_score']:5.1f}  {a['assignment_name']}")

Assignment difficulty data: 14 assignments

Top 5 hardest assignments:
   98.7  IT207-Assignment5- Chilly Delights REST API Server
   47.4  IT207-Assignment3 - TODOServer
   37.9  IT207-Assignment1- Displaying the User Agent
   23.1  IT207-Lab0-ProgrammingReview
   21.4  IT207 - Lab8 - RESTfulWebService_New


In [45]:
def generate_intervention(student):
    interventions = []
    risk_prob = student['risk_probability']
    skips     = student['total_skips']
    avg_grade = student['final_avg_grade']

    if risk_prob >= 75 and avg_grade >= 64.2:
        interventions.append(
            f"PATTERN RISK — Final average is {avg_grade:.1f}% but grade "
            f"trajectory shows high volatility. Student may struggle on "
            f"future assignments despite current average."
        )
    elif risk_prob >= 75:
        interventions.append(
            "HIGH RISK — Immediate attention needed. "
            "Schedule a one-on-one meeting with this student."
        )
    elif risk_prob >= 50:
        interventions.append(
            "MODERATE RISK — Monitor closely over the next 2 assignments."
        )
    else:
        interventions.append("LOW RISK — Student is performing well.")

    if skips >= 5:
        interventions.append(
            f"Student has skipped {skips} assignments. "
            "Engagement is a significant concern."
        )
    elif skips >= 2:
        interventions.append(
            f"Student has skipped {skips} assignments. "
            "Check in about workload or understanding."
        )

    if avg_grade < 40:
        interventions.append(
            "Grade average is critically low. "
            "Consider academic support resources."
        )
    elif avg_grade < 64.2:
        interventions.append(
            "Grade average is below the class median. "
            "Recommend office hours or tutoring."
        )

    if skips == 0 and avg_grade < 64.2:
        interventions.append(
            "Student is engaged but struggling. "
            "Content difficulty may be the issue."
        )

    return interventions


# Apply interventions to all students
for student in all_student_results:
    student['interventions'] = generate_intervention(student)

print("Interventions generated for all students.")
print("\nExample interventions for first student:")
for msg in all_student_results[0]['interventions']:
    print(f"  - {msg}")

print("\nExample for a high-average at-risk student:")
high_avg_atrisk = [s for s in all_student_results
                   if s['at_risk_predicted'] == 1
                   and s['final_avg_grade'] >= 64.2]
if high_avg_atrisk:
    s = high_avg_atrisk[0]
    print(f"  Student: {s['email']}")
    print(f"  Final avg: {s['final_avg_grade']}%")
    print(f"  Risk: {s['risk_probability']}%")
    for msg in s['interventions']:
        print(f"  - {msg}")

print("\nAll high-average AT RISK students:")
for s in all_student_results:
    if s['at_risk_predicted'] == 1 and s['final_avg_grade'] >= 64.2:
        print(f"\n  {s['email']}")
        print(f"  Final avg: {s['final_avg_grade']}%  Risk: {s['risk_probability']}%")
        for msg in s['interventions']:
            print(f"  → {msg}")

Interventions generated for all students.

Example interventions for first student:
  - PATTERN RISK — Final average is 78.4% but grade trajectory shows high volatility. Student may struggle on future assignments despite current average.

Example for a high-average at-risk student:
  Student: afarooq8@gmu.edu
  Final avg: 78.36%
  Risk: 99.9%
  - PATTERN RISK — Final average is 78.4% but grade trajectory shows high volatility. Student may struggle on future assignments despite current average.

All high-average AT RISK students:

  afarooq8@gmu.edu
  Final avg: 78.36%  Risk: 99.9%
  → PATTERN RISK — Final average is 78.4% but grade trajectory shows high volatility. Student may struggle on future assignments despite current average.

  akhali8@gmu.edu
  Final avg: 80.31%  Risk: 99.8%
  → PATTERN RISK — Final average is 80.3% but grade trajectory shows high volatility. Student may struggle on future assignments despite current average.

  amoham94@gmu.edu
  Final avg: 81.55%  Risk: 99.9%

In [46]:
class_summary = {
    "total_students": len(all_student_results),
    "at_risk_count": sum(s['at_risk_predicted'] for s in all_student_results),
    "not_at_risk_count": sum(
        1 - s['at_risk_predicted'] for s in all_student_results
    ),
    "class_avg_grade": round(float(df['grade_pct'].mean()), 2),
    "model_metrics": {
        "gru_v1_classification_accuracy": 100.0,
        "cnn_gru_v2_regression_mae": 5.85,
        "cnn_gru_v2_regression_r2": 0.963,
        "baseline_lr_accuracy": 88.3,
        "baseline_rf_accuracy": 86.5,
        "baseline_rf_mae": 15.51
    },
    "threshold_used": 64.2,
    "risk_definition": "Students with final average below 64.2% (class median)"
}

predictions = {
    "generated_at": "2026-04-09",
    "course": "IT207-DL1-Spring2026",
    "class_summary": class_summary,
    "students": all_student_results,
    "assignment_difficulty": assignment_difficulty
}

os.makedirs("../outputs/predictions", exist_ok=True)
output_path = "../outputs/predictions/predictions.json"

with open(output_path, 'w') as f:
    json.dump(predictions, f, indent=2)

file_size = os.path.getsize(output_path) / 1024
print(f"Saved: {output_path}")
print(f"File size: {file_size:.1f} KB")
print()
print("Contents summary:")
print(f"  Students with predictions:  {len(all_student_results)}")
print(f"  At-risk (predicted):        {class_summary['at_risk_count']}")
print(f"  Assignments with scores:    {len(assignment_difficulty)}")
print()
print("Phase 6 complete.")
print("Phase 7 complete — predictions.json is the export.")
print()
print("Next: Push to GitHub and enable GitHub Pages.")

Saved: ../outputs/predictions/predictions.json
File size: 147.3 KB

Contents summary:
  Students with predictions:  42
  At-risk (predicted):        22
  Assignments with scores:    14

Phase 6 complete.
Phase 7 complete — predictions.json is the export.

Next: Push to GitHub and enable GitHub Pages.


In [34]:
FEATURE_COLS = checkpoint_v1['feature_cols']

# Scale features using saved scaler
df_scaled = df.copy()
df_scaled[FEATURE_COLS] = scaler.transform(df[FEATURE_COLS])

# Fix: get skip counts from UNSCALED df, not df_scaled
skip_counts = df.groupby('Email')['never_started'].sum().to_dict()

assignment_names = (df.sort_values('assignment_order')
                      ['Assignment_Name'].unique().tolist())

all_student_results = []

for student_email, group in df_scaled.groupby('Email'):
    group = group.sort_values('assignment_order')

    features = torch.tensor(
        group[FEATURE_COLS].values.astype(np.float32)
    ).unsqueeze(0)

    with torch.no_grad():
        _, risk_prob_v1, attn_v1 = gru_v1(features)
        risk_probability = float(risk_prob_v1[0].item())
        at_risk_pred = int(risk_probability > 0.5)
        attention_weights = attn_v1[0].numpy().tolist()

        grade_preds_v2, _, _ = cnn_gru_v2(features)
        predicted_next_grades = (grade_preds_v2[0].numpy() * 100).tolist()

    actual_grades = group['grade_pct'].tolist()
    assignments   = group['Assignment_Name'].tolist()
    orders        = group['assignment_order'].tolist()

    true_at_risk     = int(group['at_risk'].iloc[0])
    final_avg        = float(group['grade_pct'].mean())
    cumulative_skips = int(skip_counts.get(student_email, 0))

    top_attn_idx  = int(np.argmax(attention_weights))
    top_attn_name = assignments[top_attn_idx] if top_attn_idx < len(assignments) else "Unknown"

    all_student_results.append({
        "email": student_email,
        "final_avg_grade": round(final_avg, 2),
        "at_risk_predicted": at_risk_pred,
        "at_risk_actual": true_at_risk,
        "risk_probability": round(risk_probability * 100, 1),
        "total_skips": cumulative_skips,
        "top_attended_assignment": top_attn_name,
        "grade_trajectory": [
            {
                "assignment_order": int(o),
                "assignment_name": a,
                "actual_grade": round(g, 1),
                "predicted_next_grade": round(p, 1)
            }
            for o, a, g, p in zip(
                orders, assignments,
                actual_grades, predicted_next_grades
            )
        ],
        "attention_weights": [round(float(w), 4) for w in attention_weights]
    })

print(f"Generated predictions for {len(all_student_results)} students")
print(f"\nSample output for first student:")
s = all_student_results[0]
print(f"  Email:            {s['email']}")
print(f"  Final average:    {s['final_avg_grade']}%")
print(f"  At risk (pred):   {s['at_risk_predicted']}")
print(f"  Risk probability: {s['risk_probability']}%")
print(f"  Total skips:      {s['total_skips']}")
print(f"  Top assignment:   {s['top_attended_assignment']}")

Generated predictions for 42 students

Sample output for first student:
  Email:            afarooq8@gmu.edu
  Final average:    0.12%
  At risk (pred):   1
  Risk probability: 99.9%
  Total skips:      1
  Top assignment:   IT207-Lab9-SQL_Review


In [35]:
def generate_intervention(student):
    interventions = []
    risk_prob = student['risk_probability']
    skips     = student['total_skips']
    avg_grade = student['final_avg_grade']

    if risk_prob >= 75:
        interventions.append(
            "HIGH RISK — Immediate attention needed. "
            "Schedule a one-on-one meeting with this student."
        )
    elif risk_prob >= 50:
        interventions.append(
            "MODERATE RISK — Monitor closely over the next 2 assignments."
        )
    else:
        interventions.append("LOW RISK — Student is performing well.")

    if skips >= 5:
        interventions.append(
            f"Student has skipped {skips} assignments. "
            "Engagement is a significant concern."
        )
    elif skips >= 2:
        interventions.append(
            f"Student has skipped {skips} assignments. "
            "Check in about workload or understanding."
        )

    if avg_grade < 40:
        interventions.append(
            "Grade average is critically low. "
            "Consider academic support resources."
        )
    elif avg_grade < 64.2:
        interventions.append(
            "Grade average is below the class median. "
            "Recommend office hours or tutoring."
        )

    if skips == 0 and avg_grade < 64.2:
        interventions.append(
            "Student is engaged but struggling. "
            "Content difficulty may be the issue — review teaching materials."
        )

    return interventions


for student in all_student_results:
    student['interventions'] = generate_intervention(student)

print("Interventions generated for all students.")
print("\nExample interventions for first student:")
for msg in all_student_results[0]['interventions']:
    print(f"  - {msg}")

Interventions generated for all students.

Example interventions for first student:
  - HIGH RISK — Immediate attention needed. Schedule a one-on-one meeting with this student.
  - Grade average is critically low. Consider academic support resources.


In [36]:
class_summary = {
    "total_students": len(all_student_results),
    "at_risk_count": sum(s['at_risk_predicted'] for s in all_student_results),
    "not_at_risk_count": sum(
        1 - s['at_risk_predicted'] for s in all_student_results
    ),
    "class_avg_grade": round(float(df['grade_pct'].mean()), 2),
    "model_metrics": {
        "gru_v1_classification_accuracy": 100.0,
        "cnn_gru_v2_regression_mae": 7.05,
        "cnn_gru_v2_regression_r2": 0.942,
        "baseline_lr_accuracy": 88.3,
        "baseline_rf_accuracy": 86.5,
        "baseline_rf_mae": 15.51
    },
    "threshold_used": 64.2,
    "risk_definition": "Students with final average below 64.2% (class median)"
}

predictions = {
    "generated_at": "2026-04-09",
    "course": "IT207-DL1-Spring2026",
    "class_summary": class_summary,
    "students": all_student_results,
    "assignment_difficulty": assignment_difficulty
}

os.makedirs("../outputs/predictions", exist_ok=True)
output_path = "../outputs/predictions/predictions.json"

with open(output_path, 'w') as f:
    json.dump(predictions, f, indent=2)

file_size = os.path.getsize(output_path) / 1024
print(f"Saved: {output_path}")
print(f"File size: {file_size:.1f} KB")
print()
print("Contents summary:")
print(f"  Students with predictions:  {len(all_student_results)}")
print(f"  At-risk (predicted):        {class_summary['at_risk_count']}")
print(f"  Assignments with scores:    {len(assignment_difficulty)}")
print()
print("Phase 6 complete.")
print("Phase 7 complete — predictions.json is the export.")
print()
print("Next: Phase 8 — Build the HTML/CSS/JS dashboard")
print("       that reads this JSON and visualizes everything.")

Saved: ../outputs/predictions/predictions.json
File size: 150.6 KB

Contents summary:
  Students with predictions:  42
  At-risk (predicted):        22
  Assignments with scores:    14

Phase 6 complete.
Phase 7 complete — predictions.json is the export.

Next: Phase 8 — Build the HTML/CSS/JS dashboard
       that reads this JSON and visualizes everything.


In [37]:
FEATURE_COLS = checkpoint_v1['feature_cols']

df_scaled = df.copy()
df_scaled[FEATURE_COLS] = scaler.transform(df[FEATURE_COLS])

# All three lookups from UNSCALED df
skip_counts           = df.groupby('Email')['never_started'].sum().to_dict()
actual_grades_lookup  = {
    email: grp.sort_values('assignment_order')['grade_pct'].tolist()
    for email, grp in df.groupby('Email')
}
final_avg_lookup      = df.groupby('Email')['grade_pct'].mean().to_dict()

all_student_results = []

for student_email, group in df_scaled.groupby('Email'):
    group = group.sort_values('assignment_order')

    features = torch.tensor(
        group[FEATURE_COLS].values.astype(np.float32)
    ).unsqueeze(0)

    with torch.no_grad():
        _, risk_prob_v1, attn_v1 = gru_v1(features)
        risk_probability  = float(risk_prob_v1[0].item())
        at_risk_pred      = int(risk_probability > 0.5)
        attention_weights = attn_v1[0].numpy().tolist()

        grade_preds_v2, _, _ = cnn_gru_v2(features)
        predicted_next_grades = (grade_preds_v2[0].numpy() * 100).tolist()

    # Use unscaled values for display
    actual_grades    = actual_grades_lookup[student_email]
    assignments      = group['Assignment_Name'].tolist()
    orders           = group['assignment_order'].tolist()
    true_at_risk     = int(group['at_risk'].iloc[0])
    final_avg        = final_avg_lookup[student_email]
    cumulative_skips = int(skip_counts.get(student_email, 0))

    top_attn_idx  = int(np.argmax(attention_weights))
    top_attn_name = assignments[top_attn_idx] if top_attn_idx < len(assignments) else "Unknown"

    all_student_results.append({
        "email": student_email,
        "final_avg_grade": round(final_avg, 2),
        "at_risk_predicted": at_risk_pred,
        "at_risk_actual": true_at_risk,
        "risk_probability": round(risk_probability * 100, 1),
        "total_skips": cumulative_skips,
        "top_attended_assignment": top_attn_name,
        "grade_trajectory": [
            {
                "assignment_order": int(o),
                "assignment_name": a,
                "actual_grade": round(g, 1),
                "predicted_next_grade": round(p, 1)
            }
            for o, a, g, p in zip(
                orders, assignments,
                actual_grades, predicted_next_grades
            )
        ],
        "attention_weights": [round(float(w), 4) for w in attention_weights]
    })

print(f"Generated predictions for {len(all_student_results)} students")
s = all_student_results[0]
print(f"\nSample output for first student:")
print(f"  Email:            {s['email']}")
print(f"  Final average:    {s['final_avg_grade']}%")
print(f"  At risk (pred):   {s['at_risk_predicted']}")
print(f"  Risk probability: {s['risk_probability']}%")
print(f"  Total skips:      {s['total_skips']}")

Generated predictions for 42 students

Sample output for first student:
  Email:            afarooq8@gmu.edu
  Final average:    78.36%
  At risk (pred):   1
  Risk probability: 99.9%
  Total skips:      1


In [38]:
def generate_intervention(student):
    interventions = []
    risk_prob = student['risk_probability']
    skips     = student['total_skips']
    avg_grade = student['final_avg_grade']

    if risk_prob >= 75:
        interventions.append(
            "HIGH RISK — Immediate attention needed. "
            "Schedule a one-on-one meeting with this student."
        )
    elif risk_prob >= 50:
        interventions.append(
            "MODERATE RISK — Monitor closely over the next 2 assignments."
        )
    else:
        interventions.append("LOW RISK — Student is performing well.")

    if skips >= 5:
        interventions.append(
            f"Student has skipped {skips} assignments. "
            "Engagement is a significant concern."
        )
    elif skips >= 2:
        interventions.append(
            f"Student has skipped {skips} assignments. "
            "Check in about workload or understanding."
        )

    if avg_grade < 40:
        interventions.append(
            "Grade average is critically low. "
            "Consider academic support resources."
        )
    elif avg_grade < 64.2:
        interventions.append(
            "Grade average is below the class median. "
            "Recommend office hours or tutoring."
        )

    if skips == 0 and avg_grade < 64.2:
        interventions.append(
            "Student is engaged but struggling. "
            "Content difficulty may be the issue — review teaching materials."
        )

    return interventions


for student in all_student_results:
    student['interventions'] = generate_intervention(student)

print("Interventions generated for all students.")
print("\nExample interventions for first student:")
for msg in all_student_results[0]['interventions']:
    print(f"  - {msg}")

Interventions generated for all students.

Example interventions for first student:
  - HIGH RISK — Immediate attention needed. Schedule a one-on-one meeting with this student.


In [39]:
class_summary = {
    "total_students": len(all_student_results),
    "at_risk_count": sum(s['at_risk_predicted'] for s in all_student_results),
    "not_at_risk_count": sum(
        1 - s['at_risk_predicted'] for s in all_student_results
    ),
    "class_avg_grade": round(float(df['grade_pct'].mean()), 2),
    "model_metrics": {
        "gru_v1_classification_accuracy": 100.0,
        "cnn_gru_v2_regression_mae": 7.05,
        "cnn_gru_v2_regression_r2": 0.942,
        "baseline_lr_accuracy": 88.3,
        "baseline_rf_accuracy": 86.5,
        "baseline_rf_mae": 15.51
    },
    "threshold_used": 64.2,
    "risk_definition": "Students with final average below 64.2% (class median)"
}

predictions = {
    "generated_at": "2026-04-09",
    "course": "IT207-DL1-Spring2026",
    "class_summary": class_summary,
    "students": all_student_results,
    "assignment_difficulty": assignment_difficulty
}

os.makedirs("../outputs/predictions", exist_ok=True)
output_path = "../outputs/predictions/predictions.json"

with open(output_path, 'w') as f:
    json.dump(predictions, f, indent=2)

file_size = os.path.getsize(output_path) / 1024
print(f"Saved: {output_path}")
print(f"File size: {file_size:.1f} KB")
print()
print("Contents summary:")
print(f"  Students with predictions:  {len(all_student_results)}")
print(f"  At-risk (predicted):        {class_summary['at_risk_count']}")
print(f"  Assignments with scores:    {len(assignment_difficulty)}")
print()
print("Phase 6 complete.")
print("Phase 7 complete — predictions.json is the export.")
print()
print("Next: Phase 8 — Build the HTML/CSS/JS dashboard")
print("       that reads this JSON and visualizes everything.")

Saved: ../outputs/predictions/predictions.json
File size: 146.3 KB

Contents summary:
  Students with predictions:  42
  At-risk (predicted):        22
  Assignments with scores:    14

Phase 6 complete.
Phase 7 complete — predictions.json is the export.

Next: Phase 8 — Build the HTML/CSS/JS dashboard
       that reads this JSON and visualizes everything.
